In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.cluster import KMeans

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

from sklearn.preprocessing import StandardScaler, RobustScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor


from sklearn.model_selection import KFold, cross_val_score
from sklearn.base import clone

import optuna

import os
import pickle
import warnings

# Suppress warnings to keep the output clean during training
warnings.filterwarnings('ignore')

tf.random.set_seed(17)
np.random.seed(17)


2025-06-21 21:08:44.076636: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-21 21:08:44.452391: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750532924.660558    4598 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750532924.717079    4598 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750532924.983575    4598 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
def create_advanced_features(df):
    """
    Create additional, more advanced features that might help the model performance.

    Args:
        df (pd.DataFrame): The input DataFrame with base features.

    Returns:
        pd.DataFrame: The DataFrame with newly engineered features.
    """
    print("\n--- Creating Advanced Features ---")
    df_enhanced = df.copy()

    # --- 2. Interaction Terms for Key Soil Properties and Climate Variables ---
    # Define sets of features for interaction
    key_soil_features = ['pH', 'soc20', 'cec20', 'BulkDensity', 'ph20']
    key_climate_features = ['bio1', 'bio12', 'lstd', 'lstn'] # Mean Annual Temp, Annual Precip, Day/Night LST
    key_spatial_features = ['lat', 'lon'] # For interaction with geographic coordinates

    # Generate interactions between selected soil features
    # Iterate through unique pairs to avoid duplicates (e.g., A_B vs B_A) and self-interactions
    for i in range(len(key_soil_features)):
        for j in range(i + 1, len(key_soil_features)):
            f1 = key_soil_features[i]
            f2 = key_soil_features[j]
            if f1 in df_enhanced.columns and f2 in df_enhanced.columns:
                df_enhanced[f'{f1}_{f2}_interaction'] = df_enhanced[f1] * df_enhanced[f2]
    print("  Soil feature interactions created.")

    # Generate interactions between soil and climate features
    for soil_f in key_soil_features:
        for climate_f in key_climate_features:
            if soil_f in df_enhanced.columns and climate_f in df_enhanced.columns:
                df_enhanced[f'{soil_f}_{climate_f}_interaction'] = df_enhanced[soil_f] * df_enhanced[climate_f]
    print("  Soil-climate feature interactions created.")

    # Generate interactions between spatial features and other key features
    # This requires 'lat' and 'lon' to be present in the DataFrame.
    # Note: If 'lat' and 'lon' are dropped during climate zone normalization, these features won't be created.
    for geo_f in key_spatial_features:
        if geo_f in df_enhanced.columns:
            for feature in ['pH', 'soc20', 'bio1', 'BulkDensity']: # Example features for geo interactions
                if feature in df_enhanced.columns:
                    df_enhanced[f'{geo_f}_{feature}_interaction'] = df_enhanced[geo_f] * df_enhanced[feature]
    print("  Geographical interaction features created.")


    # --- 3. Polynomial Features for Important Continuous Variables ---
    # Apply polynomial transformation (e.g., squaring) to capture non-linear relationships.
    # Be mindful of adding too many, which can lead to multicollinearity and overfitting.
    polynomial_features_list = [
        'pH', 'soc20', 'cec20', 'BulkDensity', 'bio1', 'bio12', 'lstd', 'lstn',
        'mdem', 'slope', 'tim', # Topographical features
        'dows', 'ls',           # Water features
        'alb', 'wp'             # Land cover features
    ]
    # Filter to only include columns that actually exist in the DataFrame
    polynomial_features_list = [f for f in polynomial_features_list if f in df_enhanced.columns]

    degree = 2 # Creating quadratic features

    for feature in polynomial_features_list:
        df_enhanced[f'{feature}_sq'] = df_enhanced[feature] ** degree
        # You could add more degrees if empirical analysis suggests a higher-order relationship
        # df_enhanced[f'{feature}_cub'] = df_enhanced[feature] ** 3
    print(f"  Polynomial features (degree {degree}) created.")


    # --- 4. Vegetation Index Combinations/Differences (Advanced) ---
    # Assuming MODIS/Landsat/Sentinel mean features are available.
    # Create differences or further ratios between related vegetation indices if they exist and make sense.
    # Example: Difference between two types of NDVI or EVI from different sensors.
    vi_pairs_for_diff = [
        ('MCD43A4_NDVI_mcd43a4_mean', 'MOD09GA_NDVI_mod09ga_mean'),
        ('MCD43A4_EVI_mcd43a4_mean', 'MOD09GA_EVI_mod09ga_mean'),
        ('MOD13Q1_NDVI_scaled_mean', 'S2_NDVI_sent2_mean')
    ]
    for vi1, vi2 in vi_pairs_for_diff:
        if vi1 in df_enhanced.columns and vi2 in df_enhanced.columns:
            df_enhanced[f'{vi1.split("_mean")[0]}_minus_{vi2.split("_mean")[0]}'] = df_enhanced[vi1] - df_enhanced[vi2]
    print("  Vegetation Index differences created.")


    # --- Final NaN Handling for Newly Created Features ---
    # After creating new features, some NaNs might appear (e.g., if original features had NaNs that weren't caught, or edge cases).
    # This step ensures all new features are imputed before returning the DataFrame.
    print("  Performing final NaN imputation for newly created features...")
    new_features_cols = [col for col in df_enhanced.columns if col not in df.columns]
    for col in new_features_cols:
        if df_enhanced[col].isnull().any():
            if df_enhanced[col].dtype in ['float64', 'int64']:
                mean_val = df_enhanced[col].mean()
                if np.isnan(mean_val): # If the entire new column became NaN (e.g., all inputs were NaN)
                    mean_val = 0 # Fallback to 0 or another sensible default
                    print(f"    Warning: Newly created feature '{col}' is entirely NaN. Filling with 0.")
                df_enhanced[col].fillna(mean_val, inplace=True)
            else:
                # For non-numeric new columns (unlikely for these features),
                # consider df_enhanced[col].mode()[0] for categorical or a specific placeholder.
                pass # For this context, we mostly expect numeric features.
    print("  Final NaN imputation for new features complete.")
    print("--- Advanced Features Creation Complete ---")
    return df_enhanced


In [3]:
def load_and_preprocess_data(path='../dataset/', target_variables=None, modis_data_dir=None, apply_scaling=True, apply_climate_normalization=True, n_climate_zones=5):
    """
    Loads the datasets and performs comprehensive preprocessing steps,
    including merging, handling missing values (NaN and Inf), climate zone normalization,
    advanced feature engineering, and general scaling.

    Args:
        path (str): The base directory where the dataset files are located.
        target_variables (list): List of target variable column names to exclude from preprocessing (e.g., from scaling).
        modis_data_dir (str, optional): Path to the directory containing processed MODIS/Landsat/Sentinel parquet files.
                                        If None, these additional datasets are not loaded.
        apply_scaling (bool): Whether to apply StandardScaler to numerical features after climate normalization.
        apply_climate_normalization (bool): Whether to apply climate zone-specific normalization.
        n_climate_zones (int): Number of clusters for KMeans for climate zone identification.

    Returns:
        tuple: A tuple containing preprocessed train_df, test_df, train_gap_df, and test_gap_df.
    """
    print("--- Loading and Preprocessing Data ---")

    # Initialize stage_transformers to store preprocessing information
    stage_transformers = {
        'imputation_means': {}, # Stores means from training data for consistent imputation
        'scaler': None, # For general scaling
        'scaled_columns': [],
        'climate_kmeans_model': None, # Stores KMeans model for climate zones
        'climate_zone_scalers': {}, # Stores StandardScaler per climate zone
    }

    # Load main datasets
    train_path = os.path.join(path, 'Train.csv')
    test_path = os.path.join(path, 'Test.csv')
    gap_train_path = os.path.join(path, 'Gap_Train.csv')
    gap_test_path = os.path.join(path, 'Gap_Test.csv')

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    train_gap_df = pd.read_csv(gap_train_path)
    test_gap_df = pd.read_csv(gap_test_path)

    # Merge BulkDensity into test_gap_df, crucial for later calculations
    test_gap_df = pd.merge(test_gap_df, test_df[['PID', 'BulkDensity']], on='PID', how='left')

    # Handle missing values in initial train/test CSVs with mean imputation
    print("Handling initial missing values (mean imputation)...")
    for column in train_df.columns:
        if train_df[column].isnull().any():
            mean_value = train_df[column].mean()
            train_df[column].fillna(mean_value, inplace=True)
            stage_transformers['imputation_means'][column] = mean_value

    for column in test_df.columns:
        if test_df[column].isnull().any():
            fill_value = stage_transformers['imputation_means'].get(column, test_df[column].mean()) # Use train mean or test mean
            test_df[column].fillna(fill_value, inplace=True)
    print("Initial missing values handled.")

    # --- Load and process multiple auxiliary datasets from parquet files ---
    if modis_data_dir:
        print(f"Loading additional data from {modis_data_dir}...")
        auxiliary_products_info = {
            'MCD43A4': {
                'prefix': 'modis',
                'cols': ['NDVI_mcd43a4', 'EVI_mcd43a4', 'SAVI_mcd43a4', 'GNDVI_mcd43a4',
                         'SR_mcd43a4', 'NDBR_mcd43a4', 'GRVI_mcd43a4', 'Brightness_Index_mcd43a4',
                         'Red_Green_Ratio_mcd43a4', 'Chlorophyll_Index_mcd43a4', 'Blue_NIR_Ratio_mcd43a4']
            },
            'MOD09GA': {
                'prefix': 'modis',
                'cols': ['sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b04', 'sur_refl_b05',
                         'sur_refl_b06', 'sur_refl_b07', 'NDVI_mod09ga', 'EVI_mod09ga', 'SAVI_mod09ga',
                         'NDWI_mod09ga', 'BSI_mod09ga', 'GEMI_mod09ga', 'ARVI_mod09ga', 'SIPI_mod09ga']
            },
            'MOD11A1': {
                'prefix': 'modis',
                'cols': ['LST_Day_1km', 'LST_Night_1km']
            },
            'MOD13Q1': {
                'prefix': 'modis',
                'cols': ['season_sin', 'season_cos', 'EVI_scaled', 'NDVI_scaled', 'SAVI_m13q1',
                         'MSAVI_m13q1', 'SR_m13q1', 'NDBR_m13q1', 'NDSWIR_m13q1', 'NDSWIR_NIR_m13q1',
                         'Brightness_Index_m13q1', 'Red_Blue_Ratio_m13q1', 'SWIR_Blue_Ratio_m13q1',
                         'Chlorophyll_Red_Edge_m13q1', 'NRI_approx_m13q1', 'PSRI_m13q1', 'SIPI_m13q1', 'MSI_m13q1']
            },
            'MOD16A2': {
                'prefix': 'modis',
                'cols': ['ET', 'PET', 'ESI_mod16a2', 'ETD_mod16a2']
            },
            'L8': {
                'prefix': 'landsat',
                'cols': ['LST_Celsius', 'NDVI_ls8', 'EVI_ls8', 'SAVI_ls8', 'NDWI_ls8', 'BSI_ls']
            },
            'S1': {
                'prefix': 'sentinel',
                'cols': ['VH_to_VV_Ratio_dB', 'VV_minus_VH_dB', 'Span_dB']
            },
            'S2': {
                'prefix': 'sentinel',
                'cols': ['NDVI_sent2', 'EVI_sent2', 'SAVI_sent2', 'NDRE1_sent2', 'CIre_sent2',
                         'NDWI_sent2', 'LSWI_sent2', 'BSI_sent2', 'NDRE2_sent2',
                         'CLOUDY_PIXEL_PERCENTAGE', 'NODATA_PIXEL_PERCENTAGE']
            }
        }

        for product_name, info in auxiliary_products_info.items():
            file_prefix = info['prefix']
            cols_to_aggregate = info['cols']

            file_path = os.path.join(modis_data_dir, f'processed_{file_prefix}_{product_name.lower()}.parquet')

            try:
                product_df = pd.read_parquet(file_path)

                if 'PID' not in product_df.columns:
                    print(f"Warning: 'PID' column not found in {product_name} data ({file_path}). Skipping merge for this product.")
                    continue

                valid_cols = [col for col in cols_to_aggregate if col in product_df.columns]
                if not valid_cols:
                    print(f"Warning: No valid columns to aggregate found in {product_name} ({file_path}). Skipping merge for this product.")
                    continue

                product_aggregated = product_df.groupby('PID')[valid_cols].mean().reset_index()
                rename_dict = {col: f'{product_name}_{col}_mean' for col in valid_cols}
                product_aggregated.rename(columns=rename_dict, inplace=True)

                train_df = pd.merge(train_df, product_aggregated, on='PID', how='left')
                test_df = pd.merge(test_df, product_aggregated, on='PID', how='left')

                for col_orig, col_renamed in rename_dict.items():
                    if col_renamed in train_df.columns:
                        if train_df[col_renamed].isnull().any():
                            mean_val_merged = train_df[col_renamed].mean()
                            train_df[col_renamed].fillna(mean_val_merged, inplace=True)
                            stage_transformers['imputation_means'][col_renamed] = mean_val_merged
                        if test_df[col_renamed].isnull().any():
                            fill_val_merged = stage_transformers['imputation_means'].get(col_renamed, test_df[col_renamed].mean())
                            test_df[col_renamed].fillna(fill_val_merged, inplace=True)
                print(f"{product_name} data loaded and merged successfully from {file_path}.")

            except FileNotFoundError:
                print(f"Error: {product_name} file not found at {file_path}. Skipping this product.")
            except Exception as e:
                print(f"An error occurred while processing {product_name} data from {file_path}: {e}. Skipping this product.")
    else:
        print("Auxiliary data directory not provided. Skipping loading of these datasets.")

    # Set default target variables if not provided
    if target_variables is None:
        target_variables = []

    # Identify numerical columns for initial NaN/Inf handling
    numerical_cols_pre_fe = [col for col in train_df.columns
                             if train_df[col].dtype in ['int64', 'float64']
                             and col != 'PID' and col != 'site'
                             and col not in target_variables]

    print("\nPerforming initial NaN/Inf check and robust imputation...")
    for col in numerical_cols_pre_fe:
        # Convert any +/- inf to NaN
        if (np.isinf(train_df[col]).any()):
            train_df[col].replace([np.inf, -np.inf], np.nan, inplace=True)
        if (np.isinf(test_df[col]).any()):
            test_df[col].replace([np.inf, -np.inf], np.nan, inplace=True)
        
        # Fill remaining NaNs using the mean from the training set (or 0 if all NaN)
        if train_df[col].isnull().any():
            mean_val = train_df[col].mean()
            if np.isnan(mean_val):
                mean_val = 0
                print(f"  Warning: Column '{col}' in train_df is entirely NaN after inf-to-nan. Filling with 0.")
            train_df[col].fillna(mean_val, inplace=True)
            stage_transformers['imputation_means'][col] = mean_val # Store for test_df
        
        if test_df[col].isnull().any():
            fill_val = stage_transformers['imputation_means'].get(col, 0)
            test_df[col].fillna(fill_val, inplace=True)
    print("Initial NaN/Inf check and imputation complete.")

    # --- Apply Advanced Feature Engineering ---
    print("\nApplying advanced feature engineering to train and test data...")
    train_df = create_advanced_features(train_df)
    test_df = create_advanced_features(test_df)
    print("Advanced feature engineering complete.")


    # --- Climate Zone Normalization (if enabled) ---
    if apply_climate_normalization and 'lat' in train_df.columns and 'lon' in train_df.columns:
        print(f"\nApplying climate zone normalization with {n_climate_zones} zones using 'lat' and 'lon'...")
        coords_train = train_df[['lat', 'lon']].values
        coords_test = test_df[['lat', 'lon']].values

        kmeans = KMeans(n_clusters=n_climate_zones, random_state=42, n_init=10)
        train_df['climate_zone'] = kmeans.fit_predict(coords_train)
        test_df['climate_zone'] = kmeans.predict(coords_test)
        stage_transformers['climate_kmeans_model'] = kmeans

        # Define features to apply climate-zone specific scaling
        # This list should include features that are sensitive to climate and were NOT
        # newly created features that might already be scaled in some way.
        # Ensure 'lat' and 'lon' are NOT in this list if they are used as features directly,
        # as normalizing them by zone might remove their raw positional information.
        climate_sensitive_features = [
            'bio1', 'bio12', 'bio15', 'bio7', 'lstd', 'lstn',
            'MOD11A1_LST_Day_1km_mean', 'MOD11A1_LST_Night_1km_mean',
            'L8_LST_Celsius_mean',
            'MCD43A4_NDVI_mcd43a4_mean', 'MCD43A4_EVI_mcd43a4_mean',
            'MOD09GA_NDVI_mod09ga_mean', 'MOD09GA_EVI_mod09ga_mean',
            'MOD13Q1_EVI_scaled_mean', 'MOD13Q1_NDVI_scaled_mean',
            'S2_NDVI_sent2_mean', 'S2_EVI_sent2_mean'
        ]
        # Filter to only include those that actually exist in the dataframe and are numerical
        numerical_cols_after_fe = [col for col in train_df.columns
                                   if train_df[col].dtype in ['int64', 'float64']
                                   and col != 'PID' and col != 'site'
                                   and col not in target_variables] # Exclude target vars

        climate_sensitive_features = [col for col in climate_sensitive_features if col in numerical_cols_after_fe]


        if climate_sensitive_features:
            for zone in sorted(train_df['climate_zone'].unique()):
                zone_mask_train = train_df['climate_zone'] == zone
                zone_mask_test = test_df['climate_zone'] == zone

                if not train_df.loc[zone_mask_train, climate_sensitive_features].empty:
                    scaler = StandardScaler()
                    train_df.loc[zone_mask_train, climate_sensitive_features] = \
                        scaler.fit_transform(train_df.loc[zone_mask_train, climate_sensitive_features])

                    if not test_df.loc[zone_mask_test, climate_sensitive_features].empty:
                        test_df.loc[zone_mask_test, climate_sensitive_features] = \
                            scaler.transform(test_df.loc[zone_mask_test, climate_sensitive_features])
                    stage_transformers['climate_zone_scalers'][zone] = scaler
                    print(f"  Normalized features in climate zone {zone}.")
                else:
                    print(f"  No data for climate zone {zone} in training data for climate-sensitive features. Skipping normalization for this zone.")
            print("Climate zone normalization complete.")
        else:
            print("No climate-sensitive features identified or present for climate zone normalization.")
        train_df.drop(columns=['climate_zone'], errors='ignore', inplace=True)
        test_df.drop(columns=['climate_zone'], errors='ignore', inplace=True)
    elif apply_climate_normalization:
        print("Skipping climate zone normalization: 'lat' or 'lon' not found in data.")


    # Identify numerical columns for final general scaling (after climate normalization if applied)
    # This list will include all original numerical features PLUS the newly created ones.
    numerical_cols_for_general_scaling = [col for col in train_df.columns
                                          if train_df[col].dtype in ['int64', 'float64']
                                          and col != 'PID' and col != 'site'
                                          and col not in target_variables]


    # Feature Scaling (General StandardScaler)
    if apply_scaling and numerical_cols_for_general_scaling:
        print("Applying general StandardScaler to all numerical features (after all FE and climate normalization)...")

        cols_to_scale = numerical_cols_for_general_scaling

        if cols_to_scale:
            scaler = StandardScaler()

            train_df[cols_to_scale] = scaler.fit_transform(train_df[cols_to_scale])
            test_df[cols_to_scale] = scaler.transform(test_df[cols_to_scale])

            stage_transformers['scaler'] = scaler
            stage_transformers['scaled_columns'] = cols_to_scale
            print(f"Scaled {len(cols_to_scale)} numerical features.")
        else:
            print("No numerical features available for general scaling.")
            stage_transformers['scaler'] = None
            stage_transformers['scaled_columns'] = []
    elif apply_scaling and not numerical_cols_for_general_scaling:
        print("No numerical features found for general scaling.")
        stage_transformers['scaler'] = None
        stage_transformers['scaled_columns'] = []
    else:
        print("General scaling not applied as per 'apply_scaling=False'.")


    print("Data loading and comprehensive preprocessing complete.")
    print(f"Final train_df shape: {train_df.shape}")
    print(f"Final test_df shape: {test_df.shape}")

    return train_df, test_df, train_gap_df, test_gap_df


In [4]:
def create_submission_file(test_df, predictions, test_gap_df, target_columns, output_filename='submission.csv'):
    """
    Generates the final submission CSV file in the required format.
    It takes the raw model predictions and transforms them into 'ID' and 'Gap' columns.

    Args:
        test_df (pd.DataFrame): The original test DataFrame (used primarily for 'PID').
        predictions (np.ndarray): The raw predictions from the model (e.g., from the VotingRegressor).
        test_gap_df (pd.DataFrame): The preprocessed test_gap_df, which should contain
                                     'Required' and 'BulkDensity' columns.
        target_columns (list): A list of target column names (e.g., ['N', 'P', 'K', ...]).
        output_filename (str): The desired name for the output CSV submission file.
    """
    print(f"\n--- Creating Submission File: {output_filename} ---")

    # Basic check to ensure predictions align with expected structure
    if predictions.shape[1] != len(target_columns):
        print(f"Warning: Prediction array columns ({predictions.shape[1]}) do not match target column count ({len(target_columns)}).")
    if predictions.shape[0] != test_df.shape[0]:
        print(f"Warning: Prediction array rows ({predictions.shape[0]}) do not match test_df rows ({test_df.shape[0]}).")

    # Convert raw predictions into a pandas DataFrame, associating them with PIDs
    predictions_df = pd.DataFrame(predictions, columns=target_columns)
    predictions_df['PID'] = test_df['PID']

    # Melt the predictions_df from wide format to long format.
    # This creates columns for 'PID', 'Nutrient' (e.g., 'N', 'P'), and 'Available_Nutrients_in_ppm'.
    submission_melted = predictions_df.melt(
        id_vars=['PID'],
        var_name='Nutrient',
        value_name='Available_Nutrients_in_ppm'
    )
    # Sort by PID for consistency and reset index
    submission_melted = submission_melted.sort_values('PID').reset_index(drop=True)

    # Merge the melted predictions with the test_gap_df.
    # This brings in the 'Required' nutrient levels and 'BulkDensity' needed for calculations.
    nutrient_df = pd.merge(test_gap_df, submission_melted, on=['PID', 'Nutrient'], how='left')

    # Calculate 'Available_Nutrients_in_kg_ha' based on the provided formula.
    # soil_depth is constant at 20 cm as per the notebook.
    soil_depth = 20  # cm
    nutrient_df['Available_Nutrients_in_kg_ha'] = (
        nutrient_df['Available_Nutrients_in_ppm'] * soil_depth * nutrient_df['BulkDensity'] * 0.1
    )

    # Calculate the 'Gap' which is the difference between 'Required' and 'Available'.
    # A positive gap means the nutrient needs to be added, negative means there's an excess.
    nutrient_df['Gap'] = nutrient_df['Required'] - nutrient_df['Available_Nutrients_in_kg_ha']

    # Create the unique 'ID' column by concatenating 'PID' and 'Nutrient'.
    nutrient_df['ID'] = nutrient_df['PID'].astype(str) + "_" + nutrient_df['Nutrient']

    # Select only the 'ID' and 'Gap' columns for the final submission file.
    final_submission_df = nutrient_df[['ID', 'Gap']]

    # Save the final DataFrame to a CSV file without the DataFrame index.
    final_submission_df.to_csv(output_filename, index=False)
    print(f"Submission file saved successfully as {output_filename}")
    print(f"First 5 rows of the generated submission file:\n", final_submission_df.head())
    print("--- Submission File Creation Complete ---")


In [5]:
def neural_network_approach(X_train, y_train, X_val, y_val, X_test_final, target_columns, model_save_dir='nn_models'):
    """
    Implements an improved neural network approach for multi-output regression.
    
    Key improvements:
    - Better architecture with batch normalization
    - Improved learning rate scheduling
    - Better regularization strategy
    - Cross-validation for better generalization
    - Ensemble prediction for robustness
    
    Returns:
        tuple: A tuple containing:
            - models (list): List of trained NN models.
            - ensemble_rmse (float): RMSE score of the ensemble on the validation set (main metric).
            - ensemble_predictions (np.ndarray): Ensemble predictions on the X_test_final data.
            - ensemble_val_pred (np.ndarray): Ensemble predictions on the X_val data.
    """
    print("\n--- Starting Improved Neural Network Approach ---")

    # Create directory for saving the model if it doesn't exist
    if not os.path.exists(model_save_dir):
        os.makedirs(model_save_dir)

    # Step 1: Data Scaling - Using RobustScaler for better outlier handling
    scaler = RobustScaler()  # More robust to outliers than StandardScaler
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_final_scaled = scaler.transform(X_test_final)
    print("Data scaled using RobustScaler for better outlier handling.")

    # Convert to numpy arrays
    y_train_np = y_train.to_numpy()
    y_val_np = y_val.to_numpy()

    # Step 2: Define improved neural network architecture
    input_shape = X_train_scaled.shape[1]
    output_dim = len(target_columns)

    def create_model():
        model = keras.Sequential([
            # Input layer: Corrected to accept a tuple (input_shape,)
            layers.Input(shape=(input_shape,)),
            
            # First hidden layer with batch normalization
            layers.Dense(512, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            
            # Second hidden layer
            layers.Dense(256, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            
            # Third hidden layer
            layers.Dense(128, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.2),
            
            # Fourth hidden layer
            layers.Dense(64, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.2),
            
            # Output layer
            layers.Dense(output_dim, activation='linear')
        ])
        return model

    # Step 3: Train multiple models for ensemble
    print("Training ensemble of 3 models...")
    models = []
    predictions_ensemble_test = [] # Predictions on X_test_final
    predictions_ensemble_val = []  # Predictions on X_val
    
    for i in range(3):
        print(f"\nTraining model {i+1}/3...")
        
        # Create model
        model = create_model()
        
        # Improved optimizer with custom learning rate
        optimizer = keras.optimizers.Adam(learning_rate=0.001)
        model.compile(optimizer=optimizer, loss='huber', metrics=['mae'])  # Huber loss is more robust
        
        # Enhanced callbacks
        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=25,
                restore_best_weights=True,
                verbose=0
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=10,
                min_lr=1e-7,
                verbose=0
            ),
            keras.callbacks.ModelCheckpoint(
                os.path.join(model_save_dir, f'best_model_{i}.keras'),
                monitor='val_loss',
                save_best_only=True,
                verbose=0
            )
        ]
        
        # Train the model
        history = model.fit(
            X_train_scaled, y_train_np,
            epochs=300,
            batch_size=64,  # Increased batch size for stability
            validation_data=(X_val_scaled, y_val_np),
            callbacks=callbacks,
            verbose=0
        )
        
        models.append(model)
        
        # Make predictions for ensemble
        predictions_ensemble_test.append(model.predict(X_test_final_scaled, verbose=0))
        predictions_ensemble_val.append(model.predict(X_val_scaled, verbose=0))
        
        # Evaluate individual model
        val_pred = model.predict(X_val_scaled, verbose=0)
        rmse = np.sqrt(mean_squared_error(y_val_np, val_pred))
        print(f"Model {i+1} Validation RMSE: {rmse:.4f}")

    # Step 4: Ensemble predictions (average of all models)
    ensemble_predictions_final = np.mean(predictions_ensemble_test, axis=0)
    ensemble_val_pred = np.mean(predictions_ensemble_val, axis=0) # Ensemble prediction on validation set
    print("Ensemble predictions calculated.")

    # Step 5: Evaluate ensemble on validation set
    ensemble_rmse = np.sqrt(mean_squared_error(y_val_np, ensemble_val_pred))
    print(f"\nEnsemble Validation RMSE: {ensemble_rmse:.4f}")
    
    # Step 6: Feature importance analysis (simple approach)
    print("\nAnalyzing feature importance...")
    feature_names = X_train.columns.tolist()
    
    # Simple feature importance based on weights of first layer
    first_layer_weights = models[0].layers[0].get_weights()[0]
    feature_importance = np.mean(np.abs(first_layer_weights), axis=1)
    
    # Display top 10 most important features
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print("Top 10 most important features:")
    print(importance_df.head(10))

    print("--- Improved Neural Network Approach Complete ---")
    return models, ensemble_rmse, ensemble_predictions_final, ensemble_val_pred


In [6]:
def anfis_approach(X_train, y_train, X_val, y_val, X_test_final, target_columns, n_rules=5, model_save_dir='anfis_models'):
    """
    Implements an ANFIS-like neural network approach for multi-output regression.
    
    Args:
        X_train (pd.DataFrame): Training features.
        y_train (pd.DataFrame): Training targets.
        X_val (pd.DataFrame): Validation features.
        y_val (pd.DataFrame): Validation targets.
        X_test_final (pd.DataFrame): Test features for making final predictions.
        target_columns (list): A list of target column names.
        n_rules (int): Number of fuzzy rules to use in the ANFIS model.
        model_save_dir (str): Directory to save the trained ANFIS models.

    Returns:
        tuple: A tuple containing:
            - models (list): List of trained ANFIS models.
            - ensemble_rmse (float): RMSE score of the ensemble on the validation set (main metric).
            - ensemble_predictions (np.ndarray): Ensemble predictions on the X_test_final data.
            - ensemble_val_pred (np.ndarray): Ensemble predictions on the X_val data.
    """
    print(f"\n--- Starting ANFIS Approach with {n_rules} rules ---")

    # Create directory for saving the model if it doesn't exist
    if not os.path.exists(model_save_dir):
        os.makedirs(model_save_dir)

    # Step 1: Data Scaling - Using RobustScaler
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_final_scaled = scaler.transform(X_test_final)
    print("Data scaled using RobustScaler for ANFIS.")

    # Convert to numpy arrays
    y_train_np = y_train.to_numpy()
    y_val_np = y_val.to_numpy()

    # Step 2: Define ANFIS-like neural network architecture
    input_dim = X_train_scaled.shape[1]
    output_dim = len(target_columns)

    def create_anfis_model():
        """Create ANFIS architecture using Keras functional API"""
        
        # Input layer: Corrected to accept a tuple (input_dim,)
        inputs = layers.Input(shape=(input_dim,))
        
        # Layer 1: Fuzzification layer
        # Create membership functions for each input variable
        fuzzy_outputs = []
        
        for rule in range(n_rules):
            # Each rule has membership functions for all inputs
            rule_outputs = []
            for i in range(input_dim):
                # Gaussian membership function parameters (learnable)
                # Need to extract each input feature individually for its own center/width
                input_feature_i = layers.Lambda(lambda x, idx=i: x[:, idx:idx+1])(inputs)

                center = layers.Dense(1, activation='linear', name=f'center_r{rule}_i{i}')(input_feature_i)
                width = layers.Dense(1, activation='softplus', name=f'width_r{rule}_i{i}')(input_feature_i) # softplus to ensure positive width
                
                # Gaussian membership function: exp(-((x - center)^2) / (width^2))
                diff = layers.Subtract()([input_feature_i, center])
                squared_diff = layers.Lambda(lambda x: tf.square(x))(diff)
                width_squared = layers.Lambda(lambda x: tf.square(x))(width)
                
                # Avoid division by zero by adding a small epsilon
                normalized_diff = layers.Lambda(lambda x: x[0] / (x[1] + tf.keras.backend.epsilon()))([squared_diff, width_squared])
                membership = layers.Lambda(lambda x: tf.exp(-x))(normalized_diff)
                rule_outputs.append(membership)
            
            # Product (AND operation) of all membership functions in this rule
            if len(rule_outputs) > 1:
                rule_strength = layers.Multiply()(rule_outputs)
            else:
                rule_strength = rule_outputs[0] # If only one input feature or one rule
            
            fuzzy_outputs.append(rule_strength)
        
        # Layer 2: Rule firing strength (already computed above)
        # Concatenate all rule strengths
        if len(fuzzy_outputs) > 1:
            rule_strengths = layers.Concatenate(axis=1)(fuzzy_outputs) # Concatenate along axis 1 (features)
        else:
            rule_strengths = fuzzy_outputs[0]
        
        # Layer 3: Normalization layer
        # Normalize rule strengths (each rule strength / sum of all rule strengths)
        sum_strengths = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1, keepdims=True))(rule_strengths)
        normalized_strengths = layers.Lambda(lambda x: x[0] / (x[1] + tf.keras.backend.epsilon()))([rule_strengths, sum_strengths])
        
        # Layer 4: Consequent layer (linear combination for each rule)
        consequent_outputs = []
        for rule in range(n_rules):
            # Linear consequent function: f_i = p_i * x1 + q_i * x2 + ... + r_i
            # Each rule has its own dense layer that processes all inputs
            consequent = layers.Dense(output_dim, activation='linear', name=f'consequent_rule_{rule}')(inputs)
            consequent_outputs.append(consequent)
        
        # Stack consequent outputs along axis=1 (n_rules dimension)
        # Shape: (batch_size, n_rules, output_dim)
        consequents = layers.Lambda(lambda x: tf.stack(x, axis=1))(consequent_outputs)
        
        # Layer 5: Output layer (weighted sum)
        # Expand normalized_strengths to (batch_size, n_rules, 1) for broadcasting
        normalized_strengths_expanded = layers.Lambda(
            lambda x: tf.expand_dims(x, axis=-1)
        )(normalized_strengths)
        
        # Element-wise multiply: (batch_size, n_rules, output_dim) * (batch_size, n_rules, 1)
        # Result: (batch_size, n_rules, output_dim)
        weighted_consequents = layers.Multiply()(
            [consequents, normalized_strengths_expanded]
        )
        
        # Sum across the rule dimension (axis=1) to get the final output for each output_dim
        # Result: (batch_size, output_dim)
        outputs = layers.Lambda(lambda x: tf.reduce_sum(x, axis=1))(weighted_consequents)
        
        model = keras.Model(inputs=inputs, outputs=outputs)
        return model
    
    # Train ensemble of ANFIS models
    print(f"Training ensemble of 3 ANFIS models with {n_rules} rules...")
    models = []
    predictions_ensemble_test = [] # Predictions on X_test_final
    predictions_ensemble_val = []  # Predictions on X_val
    
    for i in range(3):
        print(f"\nTraining ANFIS model {i+1}/3...")
        
        # Create ANFIS model
        model = create_anfis_model()
        
        # Custom optimizer with lower learning rate for stability
        optimizer = keras.optimizers.Adam(learning_rate=0.0005)
        model.compile(optimizer=optimizer, loss='huber', metrics=['mae'])
        
        # Callbacks
        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=30,
                restore_best_weights=True,
                verbose=0
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.7,
                patience=15,
                min_lr=1e-7,
                verbose=0
            ),
            keras.callbacks.ModelCheckpoint( # Add ModelCheckpoint
                # Changed from .h5 to .keras extension as per warning
                os.path.join(model_save_dir, f'best_anfis_model_{i}.keras'),
                monitor='val_loss',
                save_best_only=True,
                verbose=0
            )
        ]
        
        # Train the model
        history = model.fit(
            X_train_scaled, y_train_np,
            epochs=200,
            batch_size=32,
            validation_data=(X_val_scaled, y_val_np),
            callbacks=callbacks,
            verbose=0
        )
        
        models.append(model)
        
        # Make predictions
        predictions_ensemble_test.append(model.predict(X_test_final_scaled, verbose=0))
        predictions_ensemble_val.append(model.predict(X_val_scaled, verbose=0))
        
        # Evaluate individual model
        val_pred = model.predict(X_val_scaled, verbose=0)
        rmse = np.sqrt(mean_squared_error(y_val_np, val_pred))
        print(f"ANFIS Model {i+1} Validation RMSE: {rmse:.4f}")
    
    # Ensemble predictions
    ensemble_predictions_final = np.mean(predictions_ensemble_test, axis=0)
    ensemble_val_pred = np.mean(predictions_ensemble_val, axis=0) # Ensemble prediction on validation set
    
    # Evaluate ensemble
    ensemble_rmse = np.sqrt(mean_squared_error(y_val_np, ensemble_val_pred))
    print(f"\nANFIS Ensemble Validation RMSE: {ensemble_rmse:.4f}")
    
    # Optional: Feature importance for ANFIS (more complex than simple NNs)
    # This would involve analyzing the fuzzy rules and their consequents.
    # For now, skipping direct feature importance from the ANFIS architecture for simplicity.
    
    print("--- ANFIS Approach Complete ---")
    return models, ensemble_rmse, ensemble_predictions_final, ensemble_val_pred


In [7]:
def hybrid_anfis_nn_approach(X_train, y_train, X_val, y_val, X_test_final, target_columns, model_save_dir='hybrid_models'):
    """
    Combines ANFIS with traditional Neural Network for improved performance.
    Uses weighted ensemble of both approaches.
    
    Returns:
        tuple: A tuple containing:
            - all_models (list): List of trained ANFIS and NN models.
            - hybrid_rmse (float): RMSE score of the hybrid ensemble on the validation set.
            - hybrid_predictions (np.ndarray): Hybrid ensemble predictions on the X_test_final data.
    """
    print("\n--- Starting Hybrid ANFIS + Neural Network Approach ---")
    
    if not os.path.exists(model_save_dir):
        os.makedirs(model_save_dir)
    
    # Train ANFIS models (they perform their own scaling)
    anfis_models, anfis_rmse, anfis_predictions_test, anfis_predictions_val = anfis_approach(
        X_train, y_train, X_val, y_val, X_test_final, target_columns, model_save_dir=os.path.join(model_save_dir, 'anfis_submodels')
    )
    
    # Train Neural Network models (they perform their own scaling)
    nn_models, nn_rmse, nn_predictions_test, nn_predictions_val = neural_network_approach(
        X_train, y_train, X_val, y_val, X_test_final, target_columns, model_save_dir=os.path.join(model_save_dir, 'nn_submodels')
    )
    
    # Weighted ensemble based on validation performance
    # Better performing model gets higher weight
    # Add a small epsilon to avoid division by zero if RMSE is exactly 0
    anfis_weight = 1.0 / (anfis_rmse + 1e-8)
    nn_weight = 1.0 / (nn_rmse + 1e-8)
    total_weight = anfis_weight + nn_weight
    
    anfis_weight /= total_weight
    nn_weight /= total_weight
    
    print(f"ANFIS weight: {anfis_weight:.3f}, NN weight: {nn_weight:.3f}")
    
    # Combine predictions for the final submission
    hybrid_predictions = (anfis_weight * anfis_predictions_test + nn_weight * nn_predictions_test)
    
    # Evaluate hybrid approach on validation set using the ensemble validation predictions
    hybrid_val_pred = (anfis_weight * anfis_predictions_val + nn_weight * nn_predictions_val)
    hybrid_rmse = np.sqrt(mean_squared_error(y_val.to_numpy(), hybrid_val_pred))
    
    print(f"\nHybrid ANFIS+NN Validation RMSE: {hybrid_rmse:.4f}")
    print(f"ANFIS alone Validation RMSE: {anfis_rmse:.4f}")
    print(f"NN alone Validation RMSE: {nn_rmse:.4f}")
    
    all_models = anfis_models + nn_models # Return all individual models if needed for inspection
    
    print("--- Hybrid ANFIS + Neural Network Approach Complete ---")
    return all_models, hybrid_rmse, hybrid_predictions


In [ ]:
# --- Main execution block ---
if __name__ == "__main__":
    # Define the path to your dataset
    DATASET_PATH = '../dataset/'
    # Path to the directory containing processed auxiliary data files
    PROCESSED_DATA_DIR = '../processed_data/'

    TARGET_COLUMNS = ['N', 'P', 'K', 'Ca', 'Mg', 'S', 'Fe', 'Mn', 'Zn', 'Cu', 'B']

    # Step 1: Load and preprocess data using the enhanced function
    train_df_full, test_df_full, train_gap_df, test_gap_df = load_and_preprocess_data(
        path=DATASET_PATH,
        target_variables=TARGET_COLUMNS,
        modis_data_dir=PROCESSED_DATA_DIR,
        apply_scaling=False, # Set to False as NN/ANFIS handle scaling internally
        apply_climate_normalization=True, # Apply climate normalization
        n_climate_zones=5 # Example: use 5 climate zones
    )

    # Define base feature sets
    core_features = [
        'pH', 'ph20', 'BulkDensity', 'cec20', 'ecec20', 'hp20',
        'snd20', 'soc20', 'xhp20'
    ]
    climate_features = [
        'bio1', 'bio12', 'bio15', 'bio7', 'lstd', 'lstn'
    ]
    topographical_features = [
        'mdem', 'slope', 'tim'
    ]
    water_features = [
        'dows', 'ls'
    ]
    land_cover_features = [
        'alb', 'wp'
    ]
    modis16a2_features = [
        'MOD16A2_ET_mean', 'MOD16A2_PET_mean', 'MOD16A2_ESI_mod16a2_mean', 'MOD16A2_ETD_mod16a2_mean'
    ]
    mcd43a4_features = [
        'MCD43A4_NDVI_mcd43a4_mean', 'MCD43A4_EVI_mcd43a4_mean', 'MCD43A4_SAVI_mcd43a4_mean',
        'MCD43A4_GNDVI_mcd43a4_mean', 'MCD43A4_SR_mcd43a4', 'MCD43A4_NDBR_mcd43a4_mean',
        'MCD43A4_GRVI_mcd43a4_mean', 'MCD43A4_BrightnessIndex_mcd43a4_mean',
        'MCD43A4_Red_Green_Ratio_mcd43a4_mean', 'MCD43A4_Chlorophyll_Index_mcd43a4_mean',
        'MCD43A4_Blue_NIR_Ratio_mcd43a4_mean'
    ]
    mod09ga_features = [
        'MOD09GA_sur_refl_b01_mean', 'MOD09GA_sur_refl_b02_mean', 'MOD09GA_sur_refl_b03_mean',
        'MOD09GA_sur_refl_b04_mean', 'MOD09GA_sur_refl_b05_mean', 'MOD09GA_sur_refl_b06_mean',
        'MOD09GA_sur_refl_b07_mean', 'MOD09GA_NDVI_mod09ga_mean', 'MOD09GA_EVI_mod09ga_mean',
        'MOD09GA_SAVI_mod09ga_mean', 'MOD09GA_NDWI_mod09ga_mean', 'MOD09GA_BSI_mod09ga_mean',
        'MOD09GA_GEMI_mod09ga_mean', 'MOD09GA_ARVI_mod09ga_mean', 'MOD09GA_SIPI_mod09ga_mean'
    ]
    mod13q1_features = [
        'MOD13Q1_season_sin_mean', 'MOD13Q1_season_cos_mean', 'MOD13Q1_EVI_scaled_mean',
        'MOD13Q1_NDVI_scaled_mean', 'MOD13Q1_SAVI_m13q1_mean', 'MOD13Q1_MSAVI_m13q1_mean',
        'MOD13Q1_SR_m13q1_mean', '_meanMOD13Q1NDBR_m13q1_mean', 'MOD13Q1_NDSWIR_m13q1_mean',
        'MOD13Q1_NDSWIR_NIR_m13q1_mean', 'MOD13Q1_Brightness_Index_m13q1_mean',
        'MOD13Q1_Red_Blue_Ratio_m13q1_mean', 'MOD13Q1_SWIR_Blue_Ratio_m13q1_mean',
        'MOD13Q1_Chlorophyll_Red_Edge_m13q1_mean', 'MOD13Q1_NRI_approx_m13q1_mean',
        'MOD13Q1_PSRI_m13q1_mean', 'MOD13Q1_SIPI_m13q1_mean', 'MOD13Q1_MSI_m13q1_mean'
    ]
    landsat8_features = [
        'L8_LST_Celsius_mean', 'L8_NDVI_ls8_mean', 'L8_EVI_ls8_mean',
        'L8_SAVI_ls8_mean', 'L8_NDWI_ls8_mean', 'L8_BSI_ls8_mean'
    ]
    sentinel1_features = [
        'S1_VH_to_VV_Ratio_dB_mean', 'S1_VV_minus_VH_dB_mean', 'S1_Span_dB_mean'
    ]
    sentinel2_features = [
        'S2_NDVI_sent2_mean', 'S2_EVI_sent2_mean', 'S2_SAVI_sent2_mean',
        'S2_NDRE1_sent2_mean', 'S2_CIre_sent2_mean', 'S2_NDWI_sent2_mean',
        'S2_LSWI_sent2_mean', 'S2_BSI_sent2_mean', 'S2_NDRE2_sent2_mean',
        'S2_CLOUDY_PIXEL_PERCENTAGE_mean', 'S2_NODATA_PIXEL_PERCENTAGE_mean'
    ]

    advanced_features_generated = [
        'pH_soc20_interaction', 'pH_cec20_interaction', 'pH_BulkDensity_interaction', 'pH_ph20_interaction',
        'soc20_cec20_interaction', 'soc20_BulkDensity_interaction', 'soc20_ph20_interaction',
        'cec20_BulkDensity_interaction', 'cec20_ph20_interaction', 'BulkDensity_ph20_interaction',
        'pH_bio1_interaction', 'pH_bio12_interaction', 'pH_lstd_interaction', 'pH_lstn_interaction',
        'soc20_bio1_interaction', 'soc20_bio12_interaction', 'soc20_lstd_interaction', 'soc20_lstn_interaction',
        'cec20_bio1_interaction', 'cec20_bio12_interaction', 'cec20_lstd_interaction', 'cec20_lstn_interaction',
        'BulkDensity_bio1_interaction', 'BulkDensity_bio12_interaction', 'BulkDensity_lstd_interaction', 'BulkDensity_lstn_interaction',
        'ph20_bio1_interaction', 'ph20_bio12_interaction', 'ph20_lstd_interaction', 'ph20_lstn_interaction',
        'lat_pH_interaction', 'lat_soc20_interaction', 'lat_bio1_interaction', 'lat_BulkDensity_interaction',
        'lon_pH_interaction', 'lon_soc20_interaction', 'lon_bio1_interaction', 'lon_BulkDensity_interaction',
        'pH_sq', 'soc20_sq', 'cec20_sq', 'BulkDensity_sq', 'bio1_sq', 'bio12_sq', 'lstd_sq', 'lstn_sq',
        'mdem_sq', 'slope_sq', 'tim_sq', 'dows_sq', 'ls_sq', 'alb_sq', 'wp_sq',
        'MCD43A4_NDVI_mcd43a4_minus_MOD09GA_NDVI_mod09ga',
        'MCD43A4_EVI_mcd43a4_minus_MOD09GA_EVI_mod09ga',
        'MOD13Q1_NDVI_scaled_minus_S2_NDVI_sent2'
    ]
# Initialize a list to accumulate features for each scenario
    # current_features_for_scenario = []

    # # --- Scenario 1: Core Features (Neural Network) ---
    # print("\n--- Running Scenario 1: Neural Network with Core Features ---")
    current_features_for_scenario = [col for col in core_features if col in train_df_full.columns]
    
    # X_s1 = train_df_full[current_features_for_scenario]
    # y_s1 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s1 = test_df_full[current_features_for_scenario]

    # X_train_s1, X_val_s1, y_train_s1, y_val_s1 = train_test_split(X_s1, y_s1, test_size=0.2, random_state=42)

    # nn_models_s1, nn_ensemble_rmse_s1, nn_ensemble_predictions_s1, _ = neural_network_approach( # Get val_pred
    #     X_train_s1, y_train_s1, X_val_s1, y_val_s1, X_test_final_s1,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario1_core'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s1, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario1_core.csv'
    # )
    # print("Scenario 1 complete.")


    # --- Scenario 2: Core + Climate Features (Neural Network) ---
    # print("\n--- Running Scenario 2: Neural Network with Core + Climate Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features) if col in train_df_full.columns]

    # X_s2 = train_df_full[current_features_for_scenario]
    # y_s2 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s2 = test_df_full[current_features_for_scenario]

    # X_train_s2, X_val_s2, y_train_s2, y_val_s2 = train_test_split(X_s2, y_s2, test_size=0.2, random_state=42)

    # nn_models_s2, nn_ensemble_rmse_s2, nn_ensemble_predictions_s2, _ = neural_network_approach(
    #     X_train_s2, y_train_s2, X_val_s2, y_val_s2, X_test_final_s2,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario2_core_climate'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s2, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario2_core_climate.csv'
    # )
    # print("Scenario 2 complete.")


    # --- Scenario 3: Core + Climate + Topographical Features (Neural Network) ---
    # print("\n--- Running Scenario 3: Neural Network with Core + Climate + Topographical Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features) if col in train_df_full.columns]

    # X_s3 = train_df_full[current_features_for_scenario]
    # y_s3 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s3 = test_df_full[current_features_for_scenario]

    # X_train_s3, X_val_s3, y_train_s3, y_val_s3 = train_test_split(X_s3, y_s3, test_size=0.2, random_state=42)

    # nn_models_s3, nn_ensemble_rmse_s3, nn_ensemble_predictions_s3, _ = neural_network_approach(
    #     X_train_s3, y_train_s3, X_val_s3, y_val_s3, X_test_final_s3,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario3_core_climate_topo'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s3, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario3_core_climate_topo.csv'
    # )
    # print("Scenario 3 complete.")


    # --- Scenario 4: Core + Climate + Topographical + Water Features (Neural Network) ---
    # print("\n--- Running Scenario 4: Neural Network with Core + Climate + Topographical + Water Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features) if col in train_df_full.columns]

    # X_s4 = train_df_full[current_features_for_scenario]
    # y_s4 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s4 = test_df_full[current_features_for_scenario]

    # X_train_s4, X_val_s4, y_train_s4, y_val_s4 = train_test_split(X_s4, y_s4, test_size=0.2, random_state=17)

    # nn_models_s4, nn_ensemble_rmse_s4, nn_ensemble_predictions_s4, _ = neural_network_approach(
    #     X_train_s4, y_train_s4, X_val_s4, y_val_s4, X_test_final_s4,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario4_core_climate_topo_water'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s4, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario4_core_climate_topo_water.csv'
    # )
    # print("Scenario 4 complete.")


    # --- Scenario 5: Core + Climate + Topographical + Water + Land Cover Features (Neural Network) ---
    # print("\n--- Running Scenario 5: Neural Network with Core + Climate + Topographical + Water + Land Cover Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features) if col in train_df_full.columns]

    # X_s5 = train_df_full[current_features_for_scenario]
    # y_s5 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s5 = test_df_full[current_features_for_scenario]

    # X_train_s5, X_val_s5, y_train_s5, y_val_s5 = train_test_split(X_s5, y_s5, test_size=0.2, random_state=42)

    # nn_models_s5, nn_ensemble_rmse_s5, nn_ensemble_predictions_s5, _ = neural_network_approach(
    #     X_train_s5, y_train_s5, X_val_s5, y_val_s5, X_test_final_s5,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario5_core_climate_topo_water_landcover'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s5, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario5_core_climate_topo_water_landcover.csv'
    # )
    # print("Scenario 5 complete.")


    # # --- Scenario 6: Core + ... + MODIS 16A2 Features (Neural Network) ---
    # print("\n--- Running Scenario 6: Neural Network with Core + ... + MODIS 16A2 Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features + modis16a2_features) if col in train_df_full.columns]

    # X_s6 = train_df_full[current_features_for_scenario]
    # y_s6 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s6 = test_df_full[current_features_for_scenario]

    # X_train_s6, X_val_s6, y_train_s6, y_val_s6 = train_test_split(X_s6, y_s6, test_size=0.2, random_state=42)

    # nn_models_s6, nn_ensemble_rmse_s6, nn_ensemble_predictions_s6, _ = neural_network_approach(
    #     X_train_s6, y_train_s6, X_val_s6, y_val_s6, X_test_final_s6,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario6_plus_modis16a2'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s6, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario6_plus_modis16a2.csv'
    # )
    # print("Scenario 6 complete.")


    # # --- Scenario 7: Core + ... + MCD43A4 Features (Neural Network) ---
    # print("\n--- Running Scenario 7: Neural Network with Core + ... + MCD43A4 Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features + modis16a2_features + mcd43a4_features) if col in train_df_full.columns]

    # X_s7 = train_df_full[current_features_for_scenario]
    # y_s7 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s7 = test_df_full[current_features_for_scenario]

    # X_train_s7, X_val_s7, y_train_s7, y_val_s7 = train_test_split(X_s7, y_s7, test_size=0.2, random_state=42)

    # nn_models_s7, nn_ensemble_rmse_s7, nn_ensemble_predictions_s7, _ = neural_network_approach(
    #     X_train_s7, y_train_s7, X_val_s7, y_val_s7, X_test_final_s7,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario7_plus_mcd43a4'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s7, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario7_plus_mcd43a4.csv'
    # )
    # print("Scenario 7 complete.")


    # # --- Scenario 8: Core + ... + MOD09GA Features (Neural Network) ---
    # print("\n--- Running Scenario 8: Neural Network with Core + ... + MOD09GA Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features + modis16a2_features + mcd43a4_features + mod09ga_features) if col in train_df_full.columns]

    # X_s8 = train_df_full[current_features_for_scenario]
    # y_s8 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s8 = test_df_full[current_features_for_scenario]

    # X_train_s8, X_val_s8, y_train_s8, y_val_s8 = train_test_split(X_s8, y_s8, test_size=0.2, random_state=42)

    # nn_models_s8, nn_ensemble_rmse_s8, nn_ensemble_predictions_s8, _ = neural_network_approach(
    #     X_train_s8, y_train_s8, X_val_s8, y_val_s8, X_test_final_s8,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario8_plus_mod09ga'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s8, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario8_plus_mod09ga.csv'
    # )
    # print("Scenario 8 complete.")


    # # --- Scenario 9: Core + ... + MOD13Q1 Features (Neural Network) ---
    # print("\n--- Running Scenario 9: Neural Network with Core + ... + MOD13Q1 Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features + modis16a2_features + mcd43a4_features + mod09ga_features + mod13q1_features) if col in train_df_full.columns]

    # X_s9 = train_df_full[current_features_for_scenario]
    # y_s9 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s9 = test_df_full[current_features_for_scenario]

    # X_train_s9, X_val_s9, y_train_s9, y_val_s9 = train_test_split(X_s9, y_s9, test_size=0.2, random_state=42)

    # nn_models_s9, nn_ensemble_rmse_s9, nn_ensemble_predictions_s9, _ = neural_network_approach(
    #     X_train_s9, y_train_s9, X_val_s9, y_val_s9, X_test_final_s9,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario9_plus_mod13q1'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s9, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario9_plus_mod13q1.csv'
    # )
    # print("Scenario 9 complete.")


    # # --- Scenario 10: Core + ... + MOD11A1 Features (Neural Network) ---
    # print("\n--- Running Scenario 10: Neural Network with Core + ... + MOD11A1 Features ---")
    # current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features + modis16a2_features + mcd43a4_features + mod09ga_features + mod13q1_features + mod11a1_features) if col in train_df_full.columns]

    # X_s10 = train_df_full[current_features_for_scenario]
    # y_s10 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s10 = test_df_full[current_features_for_scenario]

    # X_train_s10, X_val_s10, y_train_s10, y_val_s10 = train_test_split(X_s10, y_s10, test_size=0.2, random_state=42)

    # nn_models_s10, nn_ensemble_rmse_s10, nn_ensemble_predictions_s10, _ = neural_network_approach(
    #     X_train_s10, y_train_s10, X_val_s10, y_val_s10, X_test_final_s10,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario10_plus_mod11a1'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s10, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario10_plus_mod11a1.csv'
    # )
    # print("Scenario 10 complete.")


    # # --- Scenario 11: Core + ... + Landsat 8 Features (Neural Network) ---
    # print("\n--- Running Scenario 11: Neural Network with Core + ... + Landsat 8 Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features + modis16a2_features + mcd43a4_features + mod09ga_features + mod13q1_features + landsat8_features) if col in train_df_full.columns]

    # X_s11 = train_df_full[current_features_for_scenario]
    # y_s11 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s11 = test_df_full[current_features_for_scenario]

    # X_train_s11, X_val_s11, y_train_s11, y_val_s11 = train_test_split(X_s11, y_s11, test_size=0.2, random_state=42)

    # nn_models_s11, nn_ensemble_rmse_s11, nn_ensemble_predictions_s11, _ = neural_network_approach(
    #     X_train_s11, y_train_s11, X_val_s11, y_val_s11, X_test_final_s11,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario11_plus_landsat8'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s11, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario11_plus_landsat8.csv'
    # )
    # print("Scenario 11 complete.")


    # # --- Scenario 12: Core + ... + Sentinel-1 Features (Neural Network) ---
    # print("\n--- Running Scenario 12: Neural Network with Core + ... + Sentinel-1 Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features + modis16a2_features + mcd43a4_features + mod09ga_features + mod13q1_features + landsat8_features + sentinel1_features) if col in train_df_full.columns]

    # X_s12 = train_df_full[current_features_for_scenario]
    # y_s12 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s12 = test_df_full[current_features_for_scenario]

    # X_train_s12, X_val_s12, y_train_s12, y_val_s12 = train_test_split(X_s12, y_s12, test_size=0.2, random_state=42)

    # nn_models_s12, nn_ensemble_rmse_s12, nn_ensemble_predictions_s12, _ = neural_network_approach(
    #     X_train_s12, y_train_s12, X_val_s12, y_val_s12, X_test_final_s12,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario12_plus_sentinel1'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s12, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario12_plus_sentinel1.csv'
    # )
    # print("Scenario 12 complete.")


    # # --- Scenario 13: Core + ... + Sentinel-2 Features (Neural Network) ---
    # print("\n--- Running Scenario 13: Neural Network with Core + ... + Sentinel-2 Features (All Features) ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features + modis16a2_features + mcd43a4_features + mod09ga_features + mod13q1_features + landsat8_features + sentinel1_features + sentinel2_features + advanced_features_generated) if col in train_df_full.columns]

    # X_s13 = train_df_full[current_features_for_scenario]
    # y_s13 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s13 = test_df_full[current_features_for_scenario]

    # X_train_s13, X_val_s13, y_train_s13, y_val_s13 = train_test_split(X_s13, y_s13, test_size=0.2, random_state=42)

    # nn_models_s13, nn_ensemble_rmse_s13, nn_ensemble_predictions_s13, _ = neural_network_approach(
    #     X_train_s13, y_train_s13, X_val_s13, y_val_s13, X_test_final_s13,
    #     TARGET_COLUMNS, model_save_dir='nn_models_scenario13_all_features'
    # )
    # create_submission_file(
    #     test_df_full, nn_ensemble_predictions_s13, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_nn_scenario13_all_features.csv'
    # )
    # print("Scenario 13 complete.")

    # # --- Scenario 14: Core Features (ANFIS) ---
    # print("\n--- Running Scenario 14: ANFIS with Core Features ---")
    current_features_for_scenario = [col for col in core_features if col in train_df_full.columns]

    # X_s14 = train_df_full[current_features_for_scenario]
    # y_s14 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s14 = test_df_full[current_features_for_scenario]

    # X_train_s14, X_val_s14, y_train_s14, y_val_s14 = train_test_split(X_s14, y_s14, test_size=0.2, random_state=42)

    # anfis_models_s14, anfis_ensemble_rmse_s14, anfis_ensemble_predictions_s14, _ = anfis_approach(
    #     X_train_s14, y_train_s14, X_val_s14, y_val_s14, X_test_final_s14,
    #     TARGET_COLUMNS, n_rules=10, model_save_dir='anfis_models_scenario14_core'
    # )
    # create_submission_file(
    #     test_df_full, anfis_ensemble_predictions_s14, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_anfis_scenario14_core.csv'
    # )
    # print("Scenario 14 complete.")

    # # --- Scenario 15: Core + Climate Features (ANFIS) ---
    # print("\n--- Running Scenario 15: ANFIS with Core + Climate Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features) if col in train_df_full.columns]

    # X_s15 = train_df_full[current_features_for_scenario]
    # y_s15 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s15 = test_df_full[current_features_for_scenario]

    # X_train_s15, X_val_s15, y_train_s15, y_val_s15 = train_test_split(X_s15, y_s15, test_size=0.2, random_state=42)

    # anfis_models_s15, anfis_ensemble_rmse_s15, anfis_ensemble_predictions_s15, _ = anfis_approach(
    #     X_train_s15, y_train_s15, X_val_s15, y_val_s15, X_test_final_s15,
    #     TARGET_COLUMNS, n_rules=10, model_save_dir='anfis_models_scenario15_core_climate'
    # )
    # create_submission_file(
    #     test_df_full, anfis_ensemble_predictions_s15, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_anfis_scenario15_core_climate.csv'
    # )
    # print("Scenario 15 complete.")

    # # --- Scenario 16: Core + Climate + Topographical Features (ANFIS) ---
    # print("\n--- Running Scenario 16: ANFIS with Core + Climate + Topographical Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features) if col in train_df_full.columns]

    # X_s16 = train_df_full[current_features_for_scenario]
    # y_s16 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s16 = test_df_full[current_features_for_scenario]

    # X_train_s16, X_val_s16, y_train_s16, y_val_s16 = train_test_split(X_s16, y_s16, test_size=0.2, random_state=42)

    # anfis_models_s16, anfis_ensemble_rmse_s16, anfis_ensemble_predictions_s16, _ = anfis_approach(
    #     X_train_s16, y_train_s16, X_val_s16, y_val_s16, X_test_final_s16,
    #     TARGET_COLUMNS, n_rules=10, model_save_dir='anfis_models_scenario16_core_climate_topo'
    # )
    # create_submission_file(
    #     test_df_full, anfis_ensemble_predictions_s16, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_anfis_scenario16_core_climate_topo.csv'
    # )
    # print("Scenario 16 complete.")

    # # --- Scenario 17: All Features (ANFIS) ---
    # print("\n--- Running Scenario 17: ANFIS with All Features ---")
    current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features + modis16a2_features + mcd43a4_features + mod09ga_features + mod13q1_features + landsat8_features + sentinel1_features + sentinel2_features + advanced_features_generated) if col in train_df_full.columns]

    X_s17 = train_df_full[current_features_for_scenario]
    y_s17 = train_df_full[TARGET_COLUMNS]
    X_test_final_s17 = test_df_full[current_features_for_scenario]

    X_train_s17, X_val_s17, y_train_s17, y_val_s17 = train_test_split(X_s17, y_s17, test_size=0.2, random_state=42)

    anfis_models_s17, anfis_ensemble_rmse_s17, anfis_ensemble_predictions_s17, _ = anfis_approach(
        X_train_s17, y_train_s17, X_val_s17, y_val_s17, X_test_final_s17,
        TARGET_COLUMNS, n_rules=10, model_save_dir='anfis_models_scenario17_all_features'
    )
    create_submission_file(
        test_df_full, anfis_ensemble_predictions_s17, test_gap_df, TARGET_COLUMNS,
        output_filename='submission_anfis_scenario17_all_features.csv'
    )
    print("Scenario 17 complete.")

    # # --- Scenario 18: Hybrid ANFIS + Neural Network (All Features) ---
    # print("\n--- Running Scenario 18: Hybrid ANFIS + Neural Network with All Features ---")
    # current_features_for_scenario = [col for col in (core_features + climate_features + topographical_features + water_features + land_cover_features + modis16a2_features + mcd43a4_features + mod09ga_features + mod13q1_features + landsat8_features + sentinel1_features + sentinel2_features + advanced_features_generated) if col in train_df_full.columns]

    # X_s18 = train_df_full[current_features_for_scenario]
    # y_s18 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s18 = test_df_full[current_features_for_scenario]

    # X_train_s18, X_val_s18, y_train_s18, y_val_s18 = train_test_split(X_s18, y_s18, test_size=0.2, random_state=42)

    # hybrid_models_s18, hybrid_rmse_s18, hybrid_predictions_s18 = hybrid_anfis_nn_approach(
    #     X_train_s18, y_train_s18, X_val_s18, y_val_s18, X_test_final_s18,
    #     TARGET_COLUMNS, model_save_dir='hybrid_models_scenario18_all_features'
    # )
    # create_submission_file(
    #     test_df_full, hybrid_predictions_s18, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_hybrid_scenario18_all_features.csv'
    # )
    # print("Scenario 18 complete.")

    # print("\nOverall script execution complete with multi-scenario Neural Network, ANFIS, and Hybrid training.")
    # print("Check respective directories for saved models and submission files for each scenario.")


--- Loading and Preprocessing Data ---
Handling initial missing values (mean imputation)...
Initial missing values handled.
Loading additional data from ../processed_data/...
MCD43A4 data loaded and merged successfully from ../processed_data/processed_modis_mcd43a4.parquet.
MOD09GA data loaded and merged successfully from ../processed_data/processed_modis_mod09ga.parquet.
MOD13Q1 data loaded and merged successfully from ../processed_data/processed_modis_mod13q1.parquet.
MOD16A2 data loaded and merged successfully from ../processed_data/processed_modis_mod16a2.parquet.
L8 data loaded and merged successfully from ../processed_data/processed_landsat_l8.parquet.
S1 data loaded and merged successfully from ../processed_data/processed_sentinel_s1.parquet.
S2 data loaded and merged successfully from ../processed_data/processed_sentinel_s2.parquet.

Performing initial NaN/Inf check and robust imputation...
Initial NaN/Inf check and imputation complete.

Applying advanced feature engineering to

I0000 00:00:1750532939.629209    4598 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6166 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1750533553.658681    5022 service.cc:152] XLA service 0x7d99e00034c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750533553.658716    5022 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2025-06-21 21:19:27.822726: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1750533626.782020    5022 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1750534013.084714    5022 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


ANFIS Model 1 Validation RMSE: 2014.1006

Training ANFIS model 2/3...


In [ ]:
train_df_full.columns

Index(['site', 'PID', 'lon', 'lat', 'pH', 'alb', 'bio1', 'bio12', 'bio15',
       'bio7',
       ...
       'mdem_sq', 'slope_sq', 'tim_sq', 'dows_sq', 'ls_sq', 'alb_sq', 'wp_sq',
       'MCD43A4_NDVI_mcd43a4_minus_MOD09GA_NDVI_mod09ga',
       'MCD43A4_EVI_mcd43a4_minus_MOD09GA_EVI_mod09ga',
       'MOD13Q1_NDVI_scaled_minus_S2_NDVI_sent2'],
      dtype='object', length=167)